## Model 3 Use Dual Learning Rates

Model 1 uses a single learning rate α for all outcomes. This assumes the agent learns at the same speed whether the outcome is safe (no sound) or dangerous (aversive sound). But there is no reason this should be true, and in fact there are strong theoretical reasons to expect it differs, especially in anxiety.

Model 3 replaces the single learning rate with two separate rates:

$$V_i^{(t+1)} = V_i^{(t)} + \left[(1 - o^{(t)}) \cdot \alpha^+ + o^{(t)} \cdot \alpha^-\right](o^{(t)} - V_i^{(t)})$$

- $\alpha^+$ is the learning rate when the outcome is neutral (o = 0). It controls how fast the agent learns "this stimulus is safe."
- $\alpha^-$ is the learning rate when the outcome is aversive (o = 1). It controls how fast the agent learns "this stimulus is dangerous."

On each trial, only one learning rate is active, depending on the outcome.

### Why is this important for anxiety?

This model directly tests the two leading computational theories of anxiety:

1. **Threat hypersensitivity.** If anxious participants have a higher α⁻, they update their beliefs more strongly after aversive events. They learn "danger" faster than non-anxious participants.

2. **Impaired safety learning.** If anxious participants have a lower α⁺, they are slower to update after safe outcomes. Even after repeated safe experiences, they remain fearful.

In [ ]:
def simulate_model3(alpha_p, alpha_m, beta, n_trials=N_TRIALS, v0=V0):
    #Simulate Model 3 use dual learning rates
    Va, Vb = v0, v0
    ch, ou = [], []
    for t in range(n_trials):
        p_a = np.exp(-beta*Va) / (np.exp(-beta*Va) + np.exp(-beta*Vb))
        c = 1 if np.random.rand() < p_a else 0
        block = min(t // BLOCK_SIZE, 3)
        o = 1 if np.random.rand() < (PROB_A[block] if c==1 else 1-PROB_A[block]) else 0
        lr = (1-o)*alpha_p + o*alpha_m
        if c==1: Va += lr*(o - Va)
        else:    Vb += lr*(o - Vb)
        ch.append(c); ou.append(o)
    return np.array(ch), np.array(ou)


def nll_model3(params, choices, outcomes):
    # NLL for Model 3
    alpha_p, alpha_m, beta = params
    Va, Vb = V0, V0
    nll = 0.0
    for t in range(len(choices)):
        exp_a = np.exp(-beta*Va); exp_b = np.exp(-beta*Vb)
        p_a = exp_a / (exp_a + exp_b)
        nll -= np.log(max(p_a, 1e-12)) if choices[t]==1 else np.log(max(1-p_a, 1e-12))
        lr = (1-outcomes[t])*alpha_p + outcomes[t]*alpha_m
        if choices[t]==1: Va += lr*(outcomes[t] - Va)
        else:             Vb += lr*(outcomes[t] - Vb)
    return nll

print("Model 3() defined")

In [ ]:
# Fit Model 3
x0_m3 = [0.25, 0.35, 6]
fitted_m3 = np.array([fit_model(nll_model3, choices[i], outcomes[i], x0_m3).x
                       for i in range(N_PARTICIPANTS)])

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for idx, name in enumerate(["α⁺ (neutral)", "α⁻ (aversive)", "β"]):
    axes[idx].bar(range(N_PARTICIPANTS), fitted_m3[:, idx], color=colors, alpha=0.7)
    axes[idx].axvline(24.5, color="black", ls="--", lw=1); axes[idx].set_xlabel("Participant")
    axes[idx].set_ylabel(name); axes[idx].set_title(f"Model 3: {name}")
plt.tight_layout(); plt.savefig("figures/task_k_m3.png"); plt.show()

In [ ]:
# Group comparison
print("Model 3 Group comparison")
for idx, name in enumerate(["α⁺ (neutral)", "α⁻ (aversive)", "β"]):
    t, p = ttest_ind(fitted_m3[:25, idx], fitted_m3[25:, idx], equal_var=False)
    sig = "***" if p<0.001 else "**" if p<0.01 else "*" if p<0.05 else "n.s."
    print(f"  {name}:  t = {t:.4f},  p = {p:.4f}  {sig}")